# 07 — Final Model Comparison: Indian AQI Prediction and Forecasting

**Project:** Indian Air Quality Index (AQI) analysis and prediction  
**Purpose:** Integrate and interpret three completed modelling approaches without making misleading cross-task comparisons.

| Experiment | Training data | Prediction target | Inputs | Appropriate comparison |
|---|---|---|---|---|
| **Model A: same-day estimation** | Historical city-day data (nominally 2015–2020) | AQI on the same day | Pollutants including PM2.5, PM10, NO₂, SO₂, CO, O₃ | Compare **within Model A only** |
| **Model A: next-day extension** | Historical city-day data | Tomorrow's AQI | Current-day pollutants and AQI history | Compare **within its own test set** |
| **Recent-data models** | Recent daily AQI data (actual cleaned data may include 2022–2025) | Tomorrow's AQI / category | Lagged AQI, calendar, city, context | Compare **within recent test set** |
| **Model B: cross-period** | Older historical AQI data, tested on recent years | Tomorrow's AQI | Lagged AQI, calendar, city | Historical vs recent fitting **on exactly the same test observations** |

**Research questions.** Which approaches outperform their own baselines? Does a historical AQI forecasting model transfer to recent years? Which cities present the largest forecasting errors? What are the limits of the comparisons?

**Interpretation rule:** The numeric scores from Model A's same-day AQI estimation are **not** directly comparable to next-day forecasting scores. Even two next-day scores should only be ranked directly when target definitions and test observations match.

This notebook reads existing result files; it **does not refit models**, modify other notebooks, or claim that different historical datasets have equivalent test conditions.

## 1. Setup, inputs, and reporting folders

Run this notebook from the project root or from `notebooks/`. Paths below match the uploaded source notebooks. If your repository uses different locations, edit the `SOURCES` dictionary.

Missing optional result files are reported explicitly rather than being replaced with invented numbers. No existing generated files are overwritten unless they have the same specific names as this notebook's final report outputs.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

FIGURES = ROOT / 'reports' / 'figures' / 'final_comparison'
RESULTS = ROOT / 'reports' / 'results' / 'final_comparison'
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'model_a_regression': ROOT / 'outputs' / 'regression_results.csv',
    'model_a_classification': ROOT / 'outputs' / 'classification_results.csv',
    'model_a_next_day': ROOT / 'outputs' / 'next_day_results.csv',
    'recent_regression': ROOT / 'reports' / 'results_regression_2023_25.csv',
    'recent_classification': ROOT / 'reports' / 'results_classification_2023_25.csv',
    'model_b_historical': ROOT / 'reports' / 'results' / 'modelb' / 'historical_model_results.csv',
    'model_b_matched': ROOT / 'reports' / 'results' / 'modelb' / 'all_models_same_test_results.csv',
    'model_b_predictions': ROOT / 'reports' / 'results' / 'modelb' / 'predictions_same_test_rows.csv',
    'model_b_cities': ROOT / 'reports' / 'results' / 'modelb' / 'city_wise_errors.csv',
    'recent_test_predictions': ROOT / 'data' / 'Cleaned' / 'predictions_2023_25_test.csv',
}

print('Repository:', ROOT)
print('Output figures:', FIGURES)
print('Output tables:', RESULTS)
for key, path in SOURCES.items():
    print(f'{"FOUND" if path.is_file() else "MISSING":7} | {key:25} | {path.relative_to(ROOT)}')

Repository: D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India
Output figures: D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\reports\figures\final_comparison
Output tables: D:\IISER\7th sem\idc409(data science)\AQI_project_idc409\AQI-prediction-India\reports\results\final_comparison
FOUND   | model_a_regression        | outputs\regression_results.csv
FOUND   | model_a_classification    | outputs\classification_results.csv
FOUND   | model_a_next_day          | outputs\next_day_results.csv
FOUND   | recent_regression         | reports\results_regression_2023_25.csv
FOUND   | recent_classification     | reports\results_classification_2023_25.csv
FOUND   | model_b_historical        | reports\results\modelb\historical_model_results.csv
FOUND   | model_b_matched           | reports\results\modelb\all_models_same_test_results.csv
FOUND   | model_b_predictions       | reports\results\modelb\predictions_same_test_rows.csv
FOUND   | mode

## 2. Load results and inspect schemas

Each modelling notebook exports slightly different metric column names (for example `R2` versus `R²`, or `MacroF1` versus `F1 (macro)`). We normalize only **column labels**, not metric values. Missing values remain missing.

If a CSV is unavailable, rerun the corresponding source notebook and its result-export cell before drawing conclusions.

In [2]:
def read_optional(key):
    path = SOURCES[key]
    if not path.is_file():
        print(f'NOT AVAILABLE: {key} — {path}')
        return None
    try:
        df = pd.read_csv(path)
        if df.empty:
            print(f'EMPTY: {key} — {path}')
            return None
        print(f'Loaded {key}: {len(df)} rows; columns={list(df.columns)}')
        return df
    except Exception as exc:
        print(f'COULD NOT READ {key}: {type(exc).__name__}: {exc}')
        return None

raw = {key: read_optional(key) for key in SOURCES}

def standardized(df, classification=False):
    if df is None:
        return None
    out = df.copy()
    aliases = {
        'model': 'Model', 'MAE': 'MAE', 'RMSE': 'RMSE', 'R2': 'R2',
        'R²': 'R2', 'R^2': 'R2', 'r2': 'R2',
        'Accuracy': 'Accuracy', 'accuracy': 'Accuracy',
        'Precision (macro)': 'MacroPrecision', 'Recall (macro)': 'MacroRecall',
        'F1 (macro)': 'MacroF1', 'MacroF1': 'MacroF1',
        'macro_f1': 'MacroF1', 'Macro Precision': 'MacroPrecision',
        'Macro Recall': 'MacroRecall'
    }
    out = out.rename(columns={k:v for k,v in aliases.items() if k in out.columns})
    if 'Model' not in out.columns:
        print('WARNING: Missing Model column; cannot make model-level comparison')
        return None
    for col in (['Accuracy','MacroPrecision','MacroRecall','MacroF1'] if classification else ['MAE','RMSE','R2']):
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors='coerce')
    return out

experiment_specs = [
    ('Model A — same-day AQI', 'model_a_regression', 'Same-day AQI estimation', False),
    ('Model A — next-day AQI', 'model_a_next_day', 'Next-day AQI forecast', False),
    ('Recent — next-day AQI', 'recent_regression', 'Next-day AQI forecast', False),
    ('Model B — cross-period', 'model_b_matched', 'Next-day AQI forecast (matched recent test)', False),
]
all_reg = []
for label, key, task, _ in experiment_specs:
    frame = standardized(raw[key])
    if frame is not None:
        frame['Experiment'] = label
        frame['Task'] = task
        all_reg.append(frame)
regression = pd.concat(all_reg, ignore_index=True, sort=False) if all_reg else pd.DataFrame()
if not regression.empty:
    cols = ['Experiment','Model','Task','Training','MAE','RMSE','R2']
    display(regression[[c for c in cols if c in regression.columns]].round(3))
else:
    print('No regression files could be loaded.')

Loaded model_a_regression: 4 rows; columns=['Model', 'MAE', 'RMSE', 'R2']
Loaded model_a_classification: 4 rows; columns=['Model', 'Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)']
Loaded model_a_next_day: 3 rows; columns=['Model', 'MAE', 'RMSE', 'R2']
Loaded recent_regression: 6 rows; columns=['Model', 'MAE', 'RMSE', 'R2']
Loaded recent_classification: 6 rows; columns=['Model', 'Accuracy', 'MacroF1', 'MacroPrecision', 'MacroRecall']
Loaded model_b_historical: 4 rows; columns=['Training', 'Model', 'MAE', 'RMSE', 'R2']
Loaded model_b_matched: 7 rows; columns=['Training', 'Model', 'MAE', 'RMSE', 'R2']
Loaded model_b_predictions: 3604 rows; columns=['City', 'Date', 'target_date', 'actual', 'Persistence', 'Historical - Ridge Regression', 'Historical - Random Forest', 'Historical - Hist Gradient Boosting', 'Recent - Ridge Regression', 'Recent - Random Forest', 'Recent - Hist Gradient Boosting']
Loaded model_b_cities: 23 rows; columns=['City', 'n', 'Historical_MAE', 'Persisten

,Experiment,Model,Task,Training,MAE,RMSE,R2
0,Model A — same-day AQI,Baseline (predict mean),Same-day AQI estimation,NaN,102.224,130.034,-0.169
1,Model A — same-day AQI,Linear Regression,Same-day AQI estimation,NaN,27.802,45.866,0.855
2,Model A — same-day AQI,Decision Tree,Same-day AQI estimation,NaN,23.812,50.536,0.823
3,Model A — same-day AQI,Random Forest,Same-day AQI estimation,NaN,19.961,41.165,0.883
4,Model A — next-day AQI,Persistence (tomorrow = today),Next-day AQI forecast,NaN,26.242,59.867,0.744
5,Model A — next-day AQI,Linear Regression,Next-day AQI forecast,NaN,25.657,48.565,0.832
6,Model A — next-day AQI,Random Forest,Next-day AQI forecast,NaN,22.727,48.193,0.834
7,Recent — next-day AQI,Hist Gradient Boosting,Next-day AQI forecast,NaN,23.505,33.854,0.728
8,Recent — next-day AQI,Hist Gradient Boosting (tuned),Next-day AQI forecast,NaN,23.508,33.858,0.728
9,Recent — next-day AQI,Random Forest,Next-day AQI forecast,NaN,23.531,33.976,0.726


## 3. Model A — pollutant-based same-day AQI estimation (historical data)

**What this task answers:** Given same-day pollutant concentrations and context, how closely can a model estimate the AQI reported for that day?

AQI is derived from pollutant measurements, so strong fit in this experiment is unsurprising. It should **not** be presented as evidence that tomorrow's AQI is easy to predict. Comparisons here use Aditri's regression test data only.

**Look for:** improvement over the mean-predicting baseline, Random Forest performance, and any difference after selecting fewer pollutant features. Aditri's feature-reduction result may require reading the notebook if it is not exported to this CSV.

## 4. Next-day forecasting: Model A extension and recent-data models

These are both **forecasting** experiments, but they are not automatically directly comparable: Model A's extension uses same-day pollutants plus AQI lags, while the recent-data model uses AQI-history and temporal/context features, and their evaluation dates/cities may differ.

The tables and charts below are deliberately kept **separate**. Judge each model against its own persistence baseline before discussing cross-experiment patterns.

## 5. Classification results (AQI categories)

Classification uses discrete AQI categories instead of predicting a number. The Indian CPCB classification has **six categories**: Good, Satisfactory, Moderate, Poor, Very Poor, Severe. Review any category merging applied in the recent-data notebook before discussing results.

Report **accuracy**, **macro-precision**, **macro-recall**, and **macro-F1**. Macro-F1 is particularly useful when severe categories are rare. Compare classifiers *within* each dataset; different test periods and class distributions preclude a fair head-to-head ranking.

## 6. Model B — the valid matched-test comparison

**This is the strongest cross-period comparison in the project.** Notebook 06 trains the same AQI-history forecasting architectures on historical versus recent training sets and evaluates them on **the same recent observations**.

The CSV `all_models_same_test_results.csv` should include training-cohort labels and performance for each model. We compare historical-trained and recent-trained versions only where both occur in that file. Persistence requires no fitting and is shown separately.

**Important:** A shared file is not sufficient by itself to prove integrity; notebook 06b should separately verify row alignment, target dates, and recomputed metrics.

## 7. City-wise comparison: Delhi, Chandigarh and Mumbai

City comparisons should use predictions on the **same test rows**. Notebook 06 exports model predictions with a city column and a next-day AQI target, permitting a fair city-by-city comparison of historical training, recent training and persistence.

If a focus city has too few observations, say so rather than inferring broad conclusions. City aliases may vary; the standardization below only harmonizes common spellings.